# MisionesOnline: exploración de la fuente (T2.1)

Confirma si la API REST de WordPress es accesible, cuántas notas de Política (categoría 7) hay por mes desde el 10/12/2023, qué campos trae cada nota y qué parte son notas de agencia o reescrituras de otros medios.

Todas las requests pasan por `probe.py`: solo rutas permitidas por robots.txt, User-Agent identificable, 2 s entre requests y caché en `data/probe_cache/`. Con `PROBE_OFFLINE=1` el notebook corre solo desde la caché.

In [1]:
import json
import re
from collections import Counter
from datetime import date, timedelta

import pandas as pd

import probe
from arc import SAMPLE_DAYS, START, sample_days, text_of

p = probe.Probe()
API = "https://misionesonline.net/wp-json/wp/v2/posts"
POLITICA = 7
LAST_MONTH = date(2026, 9, 1)  # último mes completo al explorar
pd.set_option("display.max_colwidth", 90)

## 1. Acceso

Se registra el estado de robots.txt y de la API con el User-Agent del proyecto. Si alguno responde con un desafío anti-bot, el helper bloquea el host y el resto del notebook no hace más requests a MisionesOnline.

In [2]:
access = []
blocked = False
for name, url in [("robots.txt", "https://misionesonline.net/robots.txt"),
                  ("API posts", f"{API}?categories={POLITICA}&per_page=1&_fields=id")]:
    try:
        r = p.get(url)
        access.append({"recurso": name, "status": r.status, "challenge": probe.is_challenge(r), "consultado": r.fetched_at,
                       "x-wp-total": r.headers.get("X-WP-Total") or r.headers.get("x-wp-total")})
    except probe.Blocked as b:
        blocked = True
        access.append({"recurso": name, "status": b.response.status, "challenge": probe.is_challenge(b.response),
                       "consultado": b.response.fetched_at, "x-wp-total": None})
blocked = blocked or any(row["challenge"] for row in access)
pd.DataFrame(access)

,recurso,status,challenge,consultado,x-wp-total
0,robots.txt,200,False,2026-10-27T15:14:43+00:00,NaN
1,API posts,200,False,2026-10-27T15:14:47+00:00,56112


## 2. Volumen mensual

`X-WP-Total` de la categoría Política por mes (fechas en hora del sitio, UTC-3). Diciembre de 2023 cuenta desde el 10.

In [3]:
def wp_total(after, before):
    r = p.get(f"{API}?categories={POLITICA}&after={after:%Y-%m-%d}T00:00:00&before={before:%Y-%m-%d}T00:00:00&per_page=1&_fields=id")
    return int(r.headers.get("X-WP-Total") or r.headers.get("x-wp-total") or 0)

months = []
first = START
while not blocked and first <= date.today().replace(day=1):
    next_month = (first.replace(day=1) + timedelta(days=32)).replace(day=1)
    months.append({"mes": f"{first:%Y-%m}", "notas": wp_total(first, next_month), "completo": first.replace(day=1) <= LAST_MONTH})
    first = next_month
monthly = pd.DataFrame(months)
monthly

,mes,notas,completo
0,2023-12,207,True
1,2024-01,134,True
2,2024-02,143,True
3,2024-03,171,True
4,2024-04,150,True
5,2024-05,203,True
6,2024-06,166,True
7,2024-07,122,True
8,2024-08,143,True
9,2024-09,195,True


In [4]:
if not blocked:
    print("Total hasta el último mes completo:", monthly.loc[monthly["completo"], "notas"].sum())
    print("Total hasta hoy:", monthly["notas"].sum())
    print("Promedio por mes completo:", round(monthly.loc[monthly["completo"], "notas"].mean()))

Total hasta el último mes completo: 7245
Total hasta hoy: 7468
Promedio por mes completo: 213


## 3. Campos

Se leen todas las notas de Política de los días 10 y 25 de cada mes (una página de 100 por día, con autor y términos embebidos) y se mide qué columnas del esquema de `docs/def.md` se pueden llenar.

In [5]:
def day_posts(day):
    r = p.get(f"{API}?categories={POLITICA}&after={day:%Y-%m-%d}T00:00:00&before={day + timedelta(days=1):%Y-%m-%d}T00:00:00"
              f"&per_page=100&_embed=author,wp:term")
    total = int(r.headers.get("X-WP-Total") or r.headers.get("x-wp-total") or 0)
    return json.loads(r.text) if r.ok else [], total

rows = []
for day in ([] if blocked else sample_days(LAST_MONTH)):
    posts, total = day_posts(day)
    for post in posts:
        terms = post.get("_embedded", {}).get("wp:term", [])
        content = text_of(post["content"]["rendered"])
        excerpt = text_of(post["excerpt"]["rendered"])
        rows.append({
            "day": day, "day_total": total, "link": post["link"], "date": post["date"], "date_gmt": post["date_gmt"],
            "titulo": text_of(post["title"]["rendered"]), "excerpt": excerpt, "content": content,
            "cuerpo_words": len(content.split()),
            "autor": [a.get("name") for a in post.get("_embedded", {}).get("author", []) if a.get("name")],
            "categorias": [t["name"] for group in terms for t in group if t.get("taxonomy") == "category"],
            "tags": [t["name"] for group in terms for t in group if t.get("taxonomy") == "post_tag"],
        })
notes = pd.DataFrame(rows)
print("Notas leídas:", len(notes), "| días con más de 100 notas:", int((notes.groupby("day")["day_total"].first() > 100).sum()) if len(notes) else 0)

Notas leídas: 545 | días con más de 100 notas: 0


In [6]:
def auto_excerpt(row):
    # WordPress builds the excerpt from the body when none is written by hand.
    head = row["excerpt"].replace("[…]", "").strip()[:60]
    return bool(head) and row["content"].startswith(head)

if len(notes):
    notes["bajada_propia"] = ~notes.apply(auto_excerpt, axis=1) & (notes["excerpt"].str.len() > 0)
    fields = pd.DataFrame([
        ("id", "hash de link (calculado)", notes["link"].str.len().gt(0).mean()),
        ("medio", "constante", None),
        ("url", "link", notes["link"].str.len().gt(0).mean()),
        ("fecha_publicacion", "date (hora del sitio) / date_gmt", notes["date"].notna().mean()),
        ("titulo", "title.rendered", notes["titulo"].str.len().gt(0).mean()),
        ("bajada", "excerpt.rendered, solo si no es el inicio del cuerpo", notes["bajada_propia"].mean()),
        ("cuerpo", "content.rendered (HTML)", notes["cuerpo_words"].gt(0).mean()),
        ("autor", "_embedded.author[].name", notes["autor"].str.len().gt(0).mean()),
        ("seccion", "categorías (_embedded wp:term)", notes["categorias"].str.len().gt(0).mean()),
        ("tags", "tags (_embedded wp:term)", notes["tags"].str.len().gt(0).mean()),
        ("fecha_scraping", "ninguna (la pone el cliente)", None),
    ], columns=["columna", "origen", "presente"]).round(3)
    display(fields)
    display(notes["cuerpo_words"].describe().round())

,columna,origen,presente
0,id,hash de link (calculado),1.000
1,medio,constante,NaN
2,url,link,1.000
3,fecha_publicacion,date (hora del sitio) / date_gmt,1.000
4,titulo,title.rendered,1.000
5,bajada,"excerpt.rendered, solo si no es el inicio del cuerpo",0.681
6,cuerpo,content.rendered (HTML),1.000
7,autor,_embedded.author[].name,1.000
8,seccion,categorías (_embedded wp:term),1.000
9,tags,tags (_embedded wp:term),0.875


count     545.0
mean      543.0
std       349.0
min        34.0
25%       325.0
50%       462.0
75%       698.0
max      2721.0
Name: cuerpo_words, dtype: float64

## 4. Notas de agencia y reescrituras de otros medios

Señales, medidas por separado:

- **fuente**: el cuerpo termina con "Fuente: X" (en los últimos 300 caracteres).
- **firma**: el comienzo del cuerpo trae una firma de agencia ("(NA)", "(Télam)", "Noticias Argentinas", etc.).
- **nacionales**: la nota también está en la categoría Nacionales.
- **autor**: los autores más frecuentes, para ver si hay firmas genéricas como "Redacción".
- **otro dominio**: el link de la nota apunta a otro sitio (notas compartidas con sitios hermanos).

In [7]:
SOURCE = re.compile(r"Fuente\s*:\s*([^.\n]{2,40})", re.IGNORECASE)
# Drop what follows the outlet name: "INFOBAE 👉 MANTENETE…", "ÁMBITO LAS ACCIONES…", "INFOBAE LEE TAMBIÉN…".
TRAILER = re.compile(r"\s*👉.*|\s+(?:LEE TAMBIÉN:?|EL|LAS|LOS)\s.*")
BYLINE = re.compile(r"\((?:NA|Télam|DyN|NA/DyN|AP|EFE|AFP|ANSA|Reuters)\)|Noticias Argentinas|Agencia Noticias", re.IGNORECASE)

if len(notes):
    notes["fuente"] = (notes["content"].str[-300:].str.extract(SOURCE, expand=False).str.strip().str.upper()
                       .str.replace(TRAILER, "", regex=True))
    notes["firma_agencia"] = notes["content"].str[:300].str.contains(BYLINE)
    notes["nacionales"] = notes["categorias"].apply(lambda c: "Nacionales" in c)
    notes["otro_dominio"] = ~notes["link"].str.startswith("https://misionesonline.net/")
    shares = pd.Series({
        "con 'Fuente: X'": notes["fuente"].notna().mean(),
        "firma de agencia al inicio": notes["firma_agencia"].mean(),
        "categoría Nacionales": notes["nacionales"].mean(),
        "cualquiera de las anteriores": (notes["fuente"].notna() | notes["firma_agencia"] | notes["nacionales"]).mean(),
        "link a otro dominio": notes["otro_dominio"].mean(),
    }).round(3)
    display(shares.rename("proporción"))
    display(notes["fuente"].value_counts().head(15).rename("notas"))
    display(pd.crosstab(notes["nacionales"], notes["fuente"].notna(), rownames=["Nacionales"], colnames=["con fuente"]))
    display(pd.Series(Counter(a for authors in notes["autor"] for a in authors)).sort_values(ascending=False).head(10).rename("notas"))

con 'Fuente: X'                 0.114
firma de agencia al inicio      0.000
categoría Nacionales            0.127
cualquiera de las anteriores    0.226
link a otro dominio             0.007
Name: proporción, dtype: float64

fuente
INFOBAE                27
TN                     18
ÁMBITO                  7
TÉLAM                   4
INFOBAE Y ÁMBITO        1
LA NACION               1
NOTICIAS ARGENTINAS     1
MINUTO UNO              1
LA NACIÓN               1
CLARÍN                  1
Name: notas, dtype: int64

con fuente,False,True
Nacionales,,
False,422,54
True,61,8


Redacción           155
Griselda Segovia     97
Ivan Rippel          36
Alejandro Mareco     35
Evelyn Graf          33
Dario Guimaraez      30
Gustavo Perez        25
Karina Gonzalez      25
Antonela Souza       17
Manolo Britez        15
Name: notas, dtype: int64

In [8]:
if len(notes):
    by_month = notes.assign(mes=pd.to_datetime(notes["day"]).dt.to_period("M").astype(str)).groupby("mes").agg(
        notas=("link", "size"), con_fuente=("fuente", lambda s: round(s.notna().mean(), 2)),
        nacionales=("nacionales", lambda s: round(s.mean(), 2)))
    display(by_month)

,notas,con_fuente,nacionales
mes,,,
2023-12,35,0.23,0.23
2024-01,8,0.12,0.00
2024-02,8,0.62,0.00
2024-03,11,0.18,0.27
2024-04,12,0.08,0.17
2024-05,6,0.67,0.00
2024-06,14,0.43,0.00
2024-07,9,0.22,0.00
2024-08,6,0.00,0.17


## 5. Ejemplos

Cinco notas de la muestra, con el título recortado (no se publica el cuerpo).

In [9]:
if len(notes):
    display(notes.sample(5, random_state=1)[["date", "link", "fuente", "nacionales"]]
            .assign(titulo=notes["titulo"].str[:80]))
print("Requests de red en esta corrida:", p.network_requests)

,date,link,fuente,nacionales,titulo
62,2024-04-10T22:25:32,https://misionesonline.net/2024/04/10/misiones-online-hugo-passalacqua/,NaN,False,El Gobernador de Misiones Hugo Passalacqua destacó que Misiones Online fue el pr
247,2025-06-10T19:01:38,https://misionesonline.net/2025/06/10/cristina-kirchner-hablo-tras-el-fallo-de-la-corte/,NaN,True,Cristina Kirchner habló tras el fallo de la Corte: “El partido judicial le pone
142,2024-10-25T12:46:32,https://misionesonline.net/2024/10/25/presentaron-la-24-edicion-de-la-fenasoja-en-posa...,NaN,False,Presentaron la 24° edición de la Fenasoja en Posadas
107,2024-08-10T14:06:03,https://www.argentinaforestal.com/2024/08/10/a-30-anos-de-la-inclusion-de-derechos-ind...,NaN,True,A 30 años de la inclusión de derechos indígenas en la Constitución Nacional: Ric
483,2026-07-10T12:22:42,https://misionesonline.net/2026/07/10/lla-encabezo-un-pedido-de-la-oposicion-para-reti...,NaN,False,LLA encabezó un pedido de la oposición para retirar el pliego de Héctor Simón


Requests de red en esta corrida: 0
